# 03 — Explainability (SHAP)

Loads a trained pipeline (run `02_model_training_tuning.ipynb` or `scripts/run_pipeline.py` first so `models/xgboost_tuned.joblib` exists) and generates global (bar + beeswarm) and local (per-prediction waterfall) SHAP explanations via `src/dac/explainability/shap_explain.py`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import joblib
from sklearn.model_selection import train_test_split

from dac.config import CONFIG
from dac.data.loader import load_home_credit
from dac.features.engineering import engineer_home_credit_features, split_feature_columns
from dac.explainability.shap_explain import explain_model

hc_cfg = CONFIG["data"]["home_credit"]
df, _ = load_home_credit()
df = engineer_home_credit_features(df)
exclude_cols = [hc_cfg["id_col"], *hc_cfg["protected_attributes"]]
numeric_cols, categorical_cols = split_feature_columns(df, hc_cfg["target_col"], exclude_cols)
X, y = df[numeric_cols + categorical_cols], df[hc_cfg["target_col"]]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=CONFIG["seed"], stratify=y)

pipeline = joblib.load(CONFIG["paths"]["models_dir"] / "xgboost_tuned.joblib")
pipeline

In [ ]:
importance = explain_model(
    pipeline, X_train, X_test, "xgboost_tuned",
    CONFIG["paths"]["figures_dir"] / "home_credit" / "shap",
)
importance.head(20)

`EXT_SOURCE_1/2/3` (external credit-bureau scores) are expected to dominate global importance, consistent with the published Home Credit Kaggle leaderboard analyses cited in the synopsis's literature review — a useful external sanity check on the explanations.